In [ ]:
import os
import time

from pyspark.sql import SparkSession

os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'

try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
except:
    pass

for key in list(os.environ.keys()):
    if 'SPARK' in key or 'JAVA_OPTS' in key:
        del os.environ[key]

# --- 2. Cluster Configuration ---
# Format: local-cluster[num_workers, cores_per_worker, memory_per_worker_in_MB]
NUM_EXECUTORS = 2
CORES_PER_EXECUTOR = 6
MEMORY_PER_EXECUTOR_MB = 4096

MASTER_URL = f"local-cluster[{NUM_EXECUTORS}, {CORES_PER_EXECUTOR}, {MEMORY_PER_EXECUTOR_MB}]"

print(f"Running in mode: {MASTER_URL}")

sp_s = (SparkSession.builder
    .master(MASTER_URL)
    .appName("LocalClusterTest")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.executor.cores", "6")
    .config("spark.executor.instances", NUM_EXECUTORS)
    .config("spark.memory.fraction", "0.6")
    .config("spark.sql.shuffle.partitions", "4")  # For tests, less than the default 200
    .getOrCreate()
)

sp_s.sparkContext.setLogLevel("WARN")

# --- 3. Configuration Check ---
print("Session created.")
print(f"Driver Memory Config: {sp_s.conf.get('spark.driver.memory')}")
print(f"Executor Memory Config: {sp_s.conf.get('spark.executor.memory')}")

# Check the number of executors (may take a couple of seconds to start)
time.sleep(3)
num_executors = len(sp_s.sparkContext.parallelize(range(10), NUM_EXECUTORS).glom().collect())
print(f"📊 Active executors (checked via RDD): {num_executors}")

# --- 4. Distribution Test (Example) ---
# To make sure the task went to executors, not stayed on the driver
def print_executor_info(iterator):
    import os
    # Get the executor ID from the process environment variables
    executor_id = os.environ.get('SPARK_EXECUTOR_ID', 'Driver/Local')
    process_id = os.getpid()
    return [f"Executor ID: {executor_id}, PID: {process_id}"]

# Create a dataframe and apply a transformation
df = sp_s.range(0, 10, 1, 4)  # 4 partitions
result = df.rdd.mapPartitions(print_executor_info).collect()

print("\n🖥️ Where tasks were executed:")
for line in result:
    print(line)

sp_s

# AA test tutorial 
AA test is important part of randomized controlled experiment, for example AB test. 

The objectives of the AA test are to verify the assumption of uniformity of samples as a result of the applied partitioning method, to select the best partition from the available ones, and to verify the applicability of statistical criteria for checking uniformity. 

For example, there is a hypothesis about the absence of dependence of features on each other. If this hypothesis is not followed, the AA test will fail.

[Wiki AA test](https://github.com/sb-ai-lab/HypEx/wiki/%D0%90%D0%90-Test) with more detailed description of terms for AA test.

<ul>
  <li><a href="#creation-of-a-new-test-dataset-with-synthetic-data">Creation of a new test dataset with synthetic data.
  <li><a href="#one-split-of-aa-test">One split of AA test.
  <li><a href="#aa-test">AA test.
  <li><a href="#aa-test-with-stratification">AA test with stratification.
</ul>

In [1]:
from hypex import AATest
from hypex.dataset import (
    ConstGroupRole,
    Dataset,
    InfoRole,
    StratificationRole,
    TargetRole,
    TreatmentRole,
)
from hypex.utils import BackendsEnum, create_test_data


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


## Creation of a new test dataset with synthetic data. 

In order to be able to work with our data in HypEx, first we need to convert it into `dataset`. It is important to mark the data fields by assigning the appropriate `roles`:
- TargetRole: a role for columns that contain features or predictor variables. Our split will be based on them. Applied by default if the role is not specified for the column.
- TreatmentRole: a role for columns that show the treatment or intervention.
- InfoRole: a role for columns that contain information about the data, such as user IDs. 

In [ ]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    },
    data=create_test_data(num_users=10_000),
    session=sp_s,
    backend=BackendsEnum.spark
)
data

In [ ]:
data.roles

In [3]:
import random 
import pandas as pd

def generate_test_data(
    n_rows: int = 100,
    filename: str | None = None,
    return_df: bool = True
) -> pd.DataFrame | None:
    
    headers = ['user_id', 
               'signup_month', 
               'treat', 'pre_spends', 
               'post_spends', 'age', 'gender', 'industry']
    
    data = []
    current_id = 0
    
    while len(data) < n_rows:
        if current_id > 0 and current_id % 10 == 0:
            current_id += 1
            continue
            
        user_id = float(current_id)
        signup_month = float(random.randint(0, 11))
        treat = float(random.choice([0, 1]))
        
        pre_spends = random.uniform(450, 550)
        
        if treat == 0.0:
            post_spends = pre_spends * random.uniform(0.80, 0.90)
        else:
            post_spends = pre_spends * random.uniform(1.00, 1.10)
            
        age = float(random.randint(18, 70))
        gender = random.choice(['M', 'F'])
        industry = random.choice(['Logistics', 'E-commerce'])
        
        data.append([
            user_id, signup_month, treat, 
            round(pre_spends, 1), round(post_spends, 1), 
            age, gender, industry
        ])
        
        current_id += 1

    df = pd.DataFrame(data, columns=headers)
    
    if filename:
        df.to_csv(filename, index=False, encoding='utf-8')
    
    if return_df:
        return df

In [ ]:
data = Dataset(
    roles={
            "user_id": InfoRole(float),
            "treat": TreatmentRole(),
            "pre_spends": TargetRole(),
            "gender": StratificationRole(str),
        },
    data=generate_test_data(n_rows=1_000_000),
    session=sp_s,
    backend=BackendsEnum.spark
)
# for pandas
# data = Dataset(
#     roles={
#         "user_id": InfoRole(int),
#         "pre_spends": TargetRole(),
#         "post_spends": TargetRole(),
#         "gender": StratificationRole(str),
#     },
#     data=create_test_data(num_users=10_000),
#     session=sp_s,
#     backend=BackendsEnum.pandas
# )
test = AATest(n_iterations=10)
result = test.execute(data)

In [5]:
data = Dataset(
    roles={
            "user_id": InfoRole(float),
            "treat": TreatmentRole(),
            "pre_spends": TargetRole(),
            "post_spends": TargetRole(),
            "gender": StratificationRole(str),
        },
    data=generate_test_data(n_rows=1_000_000),
    # session=sp_s,
    backend=BackendsEnum.pandas
)
# for pandas
# data = Dataset(
#     roles={
#         "user_id": InfoRole(int),
#         "pre_spends": TargetRole(),
#         "post_spends": TargetRole(),
#         "gender": StratificationRole(str),
#     },
#     data=create_test_data(num_users=10_000),
#     session=sp_s,
#     backend=BackendsEnum.pandas
# )
test = AATest(n_iterations=10)
result = test.execute(data)

100%|██████████| 10/10 [01:23<00:00,  8.40s/it]


# AA Test pandas vs spark comparasion

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# 1. Функция генерации данных (вместо create_test_data)
# ============================================================
def create_mega_dataframe(n_rows: int, seed: int = 42) -> pd.DataFrame:
    """
    Генерирует тестовый DataFrame размером n_rows строк.
    Один и тот же seed гарантирует одинаковые данные для Spark и Pandas,
    чтобы бенчмарк был корректным.
    """
    rng = np.random.default_rng(seed)

    def f32(x):
        return np.asarray(x, dtype=np.float32)

    def amount(scale=1000.0, mean=0.0, sigma=1.0, zero_share=0.0):
        x = rng.lognormal(mean=mean, sigma=sigma, size=n_rows) * scale
        if zero_share:
            x[rng.random(n_rows) < zero_share] = 0.0
        return f32(x)

    def flag_avg(p=0.3):
        return rng.binomial(1, p, size=(2, n_rows)).mean(axis=0).astype(np.float64)

    epk_id = np.arange(1, n_rows + 1, dtype=np.int64)

    avg_pl_val = amount(scale=1000.0, mean=0.0, sigma=1.0, zero_share=0.10)
    sum_pl_val = f32(2.0 * avg_pl_val)
    rsdl_npv   = amount(scale=2000.0, mean=0.0, sigma=1.0, zero_share=0.05)
    cltv       = f32(sum_pl_val + rsdl_npv)

    package_type = rng.choice(
        ['Standard', 'Premium', 'SberPrime', 'SberPrime+', 'None'], size=n_rows
    ).astype('object')
    tb = rng.choice([f'TB_{i:02d}' for i in range(1, 10)], size=n_rows).astype('object')
    gender = rng.choice(['M', 'F'], size=n_rows).astype('object')
    stlmnt_type = rng.choice(['Type_A', 'Type_B', 'Type_C', 'Type_D'], size=n_rows).astype('object')
    age = f32(rng.uniform(18, 75, size=n_rows))

    data = {
        'epk_id': epk_id,

        'avg_pl_val': avg_pl_val,
        'sum_pl_val': sum_pl_val,
        'rsdl_npv':   rsdl_npv,
        'cltv':       cltv,

        'avg_sdo_vklady':       amount(50_000.0,  zero_share=0.20),
        'avg_prc_vklady':       amount(700.0,     zero_share=0.20),
        'avg_pl_vklady':        amount(1_000.0,   zero_share=0.20),

        'avg_sdo_tek_scheta':   amount(20_000.0,  zero_share=0.30),
        'avg_prc_tek_scheta':   amount(200.0,     zero_share=0.30),
        'avg_pl_tek_scheta':    amount(500.0,     zero_share=0.30),

        'avg_sdo_nakop_scheta': amount(30_000.0,  zero_share=0.40),
        'avg_prc_nakop_scheta': amount(300.0,     zero_share=0.40),
        'avg_pl_nakop_scheta':  amount(800.0,     zero_share=0.40),

        'avg_sdo_dc':           amount(15_000.0,  zero_share=0.10),
        'avg_pos_dc':           amount(20_000.0,  zero_share=0.20),
        'avg_transfers_dc':     amount(10_000.0,  zero_share=0.30),
        'avg_withdrawals_dc':   amount(5_000.0,   zero_share=0.50),
        'avg_pl_dc':            amount(300.0,     zero_share=0.20),

        'avg_debt_cc':          amount(30_000.0,  zero_share=0.60),
        'avg_pos_cc':           amount(15_000.0,  zero_share=0.70),
        'avg_transfers_cc':     amount(5_000.0,   zero_share=0.80),
        'avg_withdrawals_cc':   amount(2_000.0,   zero_share=0.80),
        'avg_pl_cc':            amount(400.0,     zero_share=0.60),

        'avg_thanks':           amount(100.0,     zero_share=0.40),
        'avg_sum_pos':          amount(35_000.0,  zero_share=0.20),

        'avg_debt_pk':          amount(100_000.0, zero_share=0.70),
        'avg_prc_pk':           amount(1_500.0,   zero_share=0.70),
        'avg_pl_pk':            amount(1_500.0,   zero_share=0.70),

        'avg_debt_zhk':         amount(2_000_000.0, zero_share=0.80),
        'avg_prc_zhk':          amount(3_000.0,     zero_share=0.80),
        'avg_pl_zhk':           amount(3_000.0,     zero_share=0.80),

        'avg_pl_broker_service': amount(200.0, zero_share=0.80),
        'avg_pl_prime':          amount(300.0, zero_share=0.80),
        'avg_pl_prime_start':    amount(150.0, zero_share=0.90),
        'avg_pl_prime_plus':     amount(400.0, zero_share=0.90),
        'avg_pl_zvuk':           amount(100.0, zero_share=0.80),
        'avg_pl_samokat':        amount(150.0, zero_share=0.70),
        'avg_pl_sbermegamarket': amount(200.0, zero_share=0.70),
        'avg_pl_sbermobile':     amount(100.0, zero_share=0.70),
        'avg_pl_packages':       amount(500.0, zero_share=0.50),

        'package_type': package_type,
        'tb': tb,
        'age': age,
        'gender': gender,
        'stlmnt_type': stlmnt_type,

        'avg_payroll_client_flag': flag_avg(0.40),
        'avg_social_client_flag':  flag_avg(0.20),
        'avg_major_client_flag':   flag_avg(0.10),

        'avg_client_income': amount(80_000.0, mean=0.0, sigma=0.8, zero_share=0.05),
    }

    return pd.DataFrame(data)


# ============================================================
# 2. Бенчмарк: Spark vs Pandas
# ============================================================
# Подберите значения под ваше железо (для больших — может не влезть в память)
num_users_list = [10_000, 50_000, 100_000, 500_000, 1_000_000]
n_iterations = 10

spark_times  = []
pandas_times = []

for n_rows in num_users_list:
    print(f"\n=== n_rows = {n_rows:,} ===")

    # Генерируем один и тот же DataFrame — чтобы честно сравнивать бекенды
    df = create_mega_dataframe(n_rows, seed=42)

    # ---------- Spark ----------
    data_spark = Dataset(
        roles={"epk_id": InfoRole()},
        data=df,
        session=sp_s,
        default_role=TargetRole(),
        backend=BackendsEnum.spark,
    )
    test = AATest(n_iterations=n_iterations)

    start = time.perf_counter()
    result_spark = test.execute(data_spark)
    elapsed_spark = time.perf_counter() - start
    spark_times.append(elapsed_spark)
    print(f"  Spark : {elapsed_spark:8.3f} s")

    # ---------- Pandas ----------
    data_pandas = Dataset(
        roles={"epk_id": InfoRole()},
        data=df,
        session=sp_s,
        default_role=TargetRole(),
        backend=BackendsEnum.pandas,
    )
    test = AATest(n_iterations=n_iterations)

    try:
        start = time.perf_counter()
        result_pandas = test.execute(data_pandas)
        elapsed_pandas = time.perf_counter() - start
        pandas_times.append(elapsed_pandas)
        print(f"  Pandas: {elapsed_pandas:8.3f} s")
    except MemoryError:
        print(f"  Pandas: OOM на {n_rows:,} строк")
        pandas_times.append(None)

    # Освобождаем память перед следующей итерацией
    del df, data_spark, data_pandas


# ============================================================
# 3. Построение графика
# ============================================================
fig, ax = plt.subplots(figsize=(11, 6))

# Фильтруем None значения (OOM) у Pandas
valid_idx = [i for i, t in enumerate(pandas_times) if t is not None]
valid_n   = [num_users_list[i] for i in valid_idx]
valid_p   = [pandas_times[i]   for i in valid_idx]
valid_s   = [spark_times[i]    for i in valid_idx]

ax.plot(valid_n, valid_s, marker='o', linewidth=2.2, label='Spark',  color='#E25822')
ax.plot(valid_n, valid_p, marker='s', linewidth=2.2, label='Pandas', color='#150458')

# Для больших значений часто лучше лог-шкала
ax.set_xscale('log')
ax.set_yscale('log')

ax.set_xlabel('num_users (log scale)', fontsize=13)
ax.set_ylabel('Execution time, s (log scale)', fontsize=13)
ax.set_title(
    f'AATest execution time: Spark vs Pandas\n'
    f'(mega-dataset, n_iterations={n_iterations})',
    fontsize=14
)
ax.legend(fontsize=12)
ax.grid(True, which='both', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('spark_vs_pandas_mega_benchmark.png', dpi=150)
plt.show()


# ============================================================
# 4. Таблица результатов
# ============================================================
print("\n=== RESULTS ===")
print(f"{'n_rows':>12} | {'Spark (s)':>10} | {'Pandas (s)':>11} | {'Pandas/Spark':>12}")
print("-" * 60)
for nu, ts, tp in zip(num_users_list, spark_times, pandas_times):
    tp_str = f"{tp:.3f}" if tp is not None else "OOM"
    speedup = f"{tp/ts:.2f}x" if (tp is not None and ts > 0) else "N/A"
    print(f"{nu:>12,} | {ts:>10.3f} | {tp_str:>11} | {speedup:>12}")

In [ ]:
# ============================================================
# 5. Выгрузка результатов в CSV
# ============================================================
import pandas as pd

results_df = pd.DataFrame({
    'num_users': num_users_list,
    'spark_time_s': spark_times,
    'pandas_time_s': [t if t is not None else 'OOM' for t in pandas_times],
    'speedup_pandas_vs_spark': [
        round(tp / ts, 2) if (tp is not None and ts > 0) else None 
        for ts, tp in zip(spark_times, pandas_times)
    ],
    'winner': [
        'Pandas' if (tp is not None and tp < ts) else 'Spark'
        for ts, tp in zip(spark_times, pandas_times)
    ]
})

results_df.to_csv('aa_spark_vs_pandas_benchmark_results.csv', index=False)
print(f"\n✅ Результаты сохранены в spark_vs_pandas_benchmark_results.csv")
print(results_df)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ============================================================
# 1. Загрузка данных из CSV
# ============================================================
df = pd.read_csv('aa_spark_vs_pandas_benchmark_results.csv')
print(df)

# Обработка возможных OOM значений в pandas_time_s
df['pandas_numeric'] = pd.to_numeric(df['pandas_time_s'], errors='coerce')

# Убираем строки, где Pandas упал (если такие будут)
df_valid = df.dropna(subset=['pandas_numeric']).reset_index(drop=True)

# ============================================================
# 2. Настройка стиля — делаем красиво
# ============================================================
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': '#333333',
    'axes.linewidth': 1.2,
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
    'figure.dpi': 150,
})

# Красивая палитра
SPARK_COLOR  = '#E8630A'   # тёплый оранжевый (Apache Spark brand)
PANDAS_COLOR = '#130754'   # глубокий синий (Pandas brand)
BG_COLOR     = '#FAFAF7'   # лёгкий кремовый фон

# ============================================================
# 3. Построение графика (ЛИНЕЙНАЯ шкала)
# ============================================================
fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

# --- Основная область под кривыми (shaded) ---
ax.fill_between(
    df_valid['num_users'],
    df_valid['spark_time_s'],
    df_valid['pandas_numeric'],
    where=df_valid['spark_time_s'] < df_valid['pandas_numeric'],
    color=SPARK_COLOR, alpha=0.12,
    label='Spark быстрее'
)
ax.fill_between(
    df_valid['num_users'],
    df_valid['spark_time_s'],
    df_valid['pandas_numeric'],
    where=df_valid['spark_time_s'] >= df_valid['pandas_numeric'],
    color=PANDAS_COLOR, alpha=0.12,
    label='Pandas быстрее'
)

# --- Линии ---
ax.plot(
    df_valid['num_users'], df_valid['spark_time_s'],
    marker='o', markersize=10, markerfacecolor='white',
    linewidth=3, label='Spark', color=SPARK_COLOR,
    markeredgewidth=2.5, markeredgecolor=SPARK_COLOR,
    zorder=5,
)
ax.plot(
    df_valid['num_users'], df_valid['pandas_numeric'],
    marker='s', markersize=10, markerfacecolor='white',
    linewidth=3, label='Pandas', color=PANDAS_COLOR,
    markeredgewidth=2.5, markeredgecolor=PANDAS_COLOR,
    zorder=5,
)

# --- Подписи значений прямо на точках ---
for x, y in zip(df_valid['num_users'], df_valid['spark_time_s']):
    ax.annotate(
        f'{y:.2f}s',
        xy=(x, y), xytext=(0, 16),
        textcoords='offset points',
        ha='center', va='bottom',
        fontsize=10, fontweight='bold', color=SPARK_COLOR,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  edgecolor=SPARK_COLOR, alpha=0.8, linewidth=1),
    )

for x, y in zip(df_valid['num_users'], df_valid['pandas_numeric']):
    ax.annotate(
        f'{y:.2f}s',
        xy=(x, y), xytext=(0, -22),
        textcoords='offset points',
        ha='center', va='top',
        fontsize=10, fontweight='bold', color=PANDAS_COLOR,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  edgecolor=PANDAS_COLOR, alpha=0.8, linewidth=1),
    )

# ============================================================
# 4. Оси и оформление
# ============================================================
# X — форматирование чисел с пробелами (100 000 вместо 100000)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x):,}'.replace(',', ' ')))
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.1f}'))

ax.set_xlabel('Количество пользователей (num_users)', fontsize=14, fontweight='bold', labelpad=12)
ax.set_ylabel('Время выполнения AATest, секунды', fontsize=14, fontweight='bold', labelpad=12)

# Title с железом
ax.set_title(
    'Производительность AATest: Spark vs Pandas\n',
    fontsize=18, fontweight='bold', color='#222222', pad=20,
)
ax.text(
    0.5, 1.02,
    '50 признаков  •  10 итераций  •  Apple M4 Pro (14 CPU, 24 GB unified)',
    transform=ax.transAxes, ha='center', va='bottom',
    fontsize=12, color='#666666', style='italic',
)

# Grid — тонкие пунктирные линии
ax.grid(True, linestyle='--', alpha=0.35, color='#888888', linewidth=0.8)
ax.set_axisbelow(True)

# Y начинается с 0
y_max = max(df_valid['spark_time_s'].max(), df_valid['pandas_numeric'].max())
ax.set_ylim(0, y_max * 1.25)

# Красивый legend
legend = ax.legend(
    loc='upper left',
    frameon=True,
    fancybox=True,
    shadow=False,
    framealpha=0.95,
    edgecolor='#CCCCCC',
    fontsize=12,
    title='Backend / Зона доминирования',
    title_fontsize=11,
)
legend.get_title().set_fontweight('bold')

# ============================================================
# 5. Добавляем инсайт-блок (annotation)
# ============================================================
# Находим точку кроссовера (где Spark обгоняет Pandas)
crossover_idx = None
for i in range(len(df_valid) - 1):
    s1, s2 = df_valid.iloc[i]['spark_time_s'], df_valid.iloc[i+1]['spark_time_s']
    p1, p2 = df_valid.iloc[i]['pandas_numeric'], df_valid.iloc[i+1]['pandas_numeric']
    if p1 < s1 and p2 > s2:
        crossover_idx = i
        break

# if crossover_idx is not None:
#     cx = df_valid.iloc[crossover_idx + 1]['num_users']
#     ax.axvline(
#         x=cx, color='#555555', linestyle=':', linewidth=1.5, alpha=0.7,
#     )
#     ax.annotate(
#         f'← Точка кроссовера\n     ~{int(cx):,} пользователей',
#         xy=(cx, y_max * 0.85), xytext=(cx * 0.5, y_max * 1.05),
#         arrowprops=dict(arrowstyle='->', color='#555555', lw=1.5),
#         ha='center', fontsize=11, color='#444444',
#         bbox=dict(boxstyle='round,pad=0.5', facecolor='#FFF8DC',
#                   edgecolor='#AAAAAA', linewidth=1.2),
#     )

plt.tight_layout()
plt.savefig('aa_benchmark_linear_beautiful.png', dpi=200, bbox_inches='tight',
            facecolor=BG_COLOR)
plt.show()

print("✅ Сохранено: aa_benchmark_linear_beautiful.png")

# AB Test pandas vs spark comparasion

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from hypex import ABTest

# ============================================================
# 1. Функция генерации датасета с treatment
# ============================================================
def create_mega_dataframe(n_rows: int, seed: int = 42) -> pd.DataFrame:
    """
    Mega-датасет на n_rows клиентов + колонка treat (0/1/2).
    seed гарантирует одинаковые данные для Spark и Pandas.
    """
    rng = np.random.default_rng(seed)

    def f32(x):
        return np.asarray(x, dtype=np.float32)

    def amount(scale=1000.0, mean=0.0, sigma=1.0, zero_share=0.0):
        x = rng.lognormal(mean=mean, sigma=sigma, size=n_rows) * scale
        if zero_share:
            x[rng.random(n_rows) < zero_share] = 0.0
        return f32(x)

    def flag_avg(p=0.3):
        return rng.binomial(1, p, size=(2, n_rows)).mean(axis=0).astype(np.float64)

    epk_id = np.arange(1, n_rows + 1, dtype=np.int64)

    avg_pl_val = amount(scale=1000.0, mean=0.0, sigma=1.0, zero_share=0.10)
    sum_pl_val = f32(2.0 * avg_pl_val)
    rsdl_npv   = amount(scale=2000.0, mean=0.0, sigma=1.0, zero_share=0.05)
    cltv       = f32(sum_pl_val + rsdl_npv)

    package_type = rng.choice(
        ['Standard', 'Premium', 'SberPrime', 'SberPrime+', 'None'], size=n_rows
    ).astype('object')
    tb = rng.choice([f'TB_{i:02d}' for i in range(1, 10)], size=n_rows).astype('object')
    gender = rng.choice(['M', 'F'], size=n_rows).astype('object')
    stlmnt_type = rng.choice(['Type_A', 'Type_B', 'Type_C', 'Type_D'], size=n_rows).astype('object')
    age = f32(rng.uniform(18, 75, size=n_rows))

    # Treatment: 3 группы (0 = control, 1/2 = варианты)
    # Равномерное распределение — ~33% в каждой группе
    treat = rng.choice([0, 1, 2], size=n_rows).astype(np.int8)

    data = {
        'epk_id': epk_id,

        'avg_pl_val': avg_pl_val,
        'sum_pl_val': sum_pl_val,
        'rsdl_npv':   rsdl_npv,
        'cltv':       cltv,

        'avg_sdo_vklady':       amount(50_000.0,  zero_share=0.20),
        'avg_prc_vklady':       amount(700.0,     zero_share=0.20),
        'avg_pl_vklady':        amount(1_000.0,   zero_share=0.20),

        'avg_sdo_tek_scheta':   amount(20_000.0,  zero_share=0.30),
        'avg_prc_tek_scheta':   amount(200.0,     zero_share=0.30),
        'avg_pl_tek_scheta':    amount(500.0,     zero_share=0.30),

        'avg_sdo_nakop_scheta': amount(30_000.0,  zero_share=0.40),
        'avg_prc_nakop_scheta': amount(300.0,     zero_share=0.40),
        'avg_pl_nakop_scheta':  amount(800.0,     zero_share=0.40),

        'avg_sdo_dc':           amount(15_000.0,  zero_share=0.10),
        'avg_pos_dc':           amount(20_000.0,  zero_share=0.20),
        'avg_transfers_dc':     amount(10_000.0,  zero_share=0.30),
        'avg_withdrawals_dc':   amount(5_000.0,   zero_share=0.50),
        'avg_pl_dc':            amount(300.0,     zero_share=0.20),

        'avg_debt_cc':          amount(30_000.0,  zero_share=0.60),
        'avg_pos_cc':           amount(15_000.0,  zero_share=0.70),
        'avg_transfers_cc':     amount(5_000.0,   zero_share=0.80),
        'avg_withdrawals_cc':   amount(2_000.0,   zero_share=0.80),
        'avg_pl_cc':            amount(400.0,     zero_share=0.60),

        'avg_thanks':           amount(100.0,     zero_share=0.40),
        'avg_sum_pos':          amount(35_000.0,  zero_share=0.20),

        'avg_debt_pk':          amount(100_000.0, zero_share=0.70),
        'avg_prc_pk':           amount(1_500.0,   zero_share=0.70),
        'avg_pl_pk':            amount(1_500.0,   zero_share=0.70),

        'avg_debt_zhk':         amount(2_000_000.0, zero_share=0.80),
        'avg_prc_zhk':          amount(3_000.0,     zero_share=0.80),
        'avg_pl_zhk':           amount(3_000.0,     zero_share=0.80),

        'avg_pl_broker_service': amount(200.0, zero_share=0.80),
        'avg_pl_prime':          amount(300.0, zero_share=0.80),
        'avg_pl_prime_start':    amount(150.0, zero_share=0.90),
        'avg_pl_prime_plus':     amount(400.0, zero_share=0.90),
        'avg_pl_zvuk':           amount(100.0, zero_share=0.80),
        'avg_pl_samokat':        amount(150.0, zero_share=0.70),
        'avg_pl_sbermegamarket': amount(200.0, zero_share=0.70),
        'avg_pl_sbermobile':     amount(100.0, zero_share=0.70),
        'avg_pl_packages':       amount(500.0, zero_share=0.50),

        'package_type': package_type,
        'tb': tb,
        'age': age,
        'gender': gender,
        'stlmnt_type': stlmnt_type,

        'avg_payroll_client_flag': flag_avg(0.40),
        'avg_social_client_flag':  flag_avg(0.20),
        'avg_major_client_flag':   flag_avg(0.10),

        'avg_client_income': amount(80_000.0, mean=0.0, sigma=0.8, zero_share=0.05),

        # Treatment
        'treat': treat,
    }

    return pd.DataFrame(data)


# ============================================================
# 2. Бенчмарк ABTest: Spark vs Pandas
# ============================================================
num_users_list = [10_000, 50_000, 100_000, 500_000, 1_000_000]
n_iterations = 5  # запусков ABTest в цикле

spark_times  = []
pandas_times = []

for n_rows in num_users_list:
    print(f"\n=== n_rows = {n_rows:,} ===")

    df = create_mega_dataframe(n_rows, seed=42)

    # ---------- Spark ----------
    data_spark = Dataset(
        roles={
            "epk_id": InfoRole(),
            "treat":  TreatmentRole(),
        },
        default_role=TargetRole(),
        data=df,
        session=sp_s,
        backend=BackendsEnum.spark,
    )

    spark_iter_times = []
    for i in range(n_iterations):
        test = ABTest()
        start = time.perf_counter()
        _ = test.execute(data_spark)
        elapsed = time.perf_counter() - start
        spark_iter_times.append(elapsed)

    # Берём медиану, чтобы отбросить выбросы (JIT-прогрев, GC)
    elapsed_spark = np.median(spark_iter_times)
    spark_times.append(elapsed_spark)
    print(f"  Spark : median={elapsed_spark:8.3f} s  (runs: {[round(x,2) for x in spark_iter_times]})")

    # ---------- Pandas ----------
    data_pandas = Dataset(
        roles={
            "epk_id": InfoRole(),
            "treat":  TreatmentRole(),
        },
        default_role=TargetRole(),
        data=df,
        session=sp_s,
        backend=BackendsEnum.pandas,
    )

    try:
        pandas_iter_times = []
        for i in range(n_iterations):
            test = ABTest()
            start = time.perf_counter()
            _ = test.execute(data_pandas)
            elapsed = time.perf_counter() - start
            pandas_iter_times.append(elapsed)

        elapsed_pandas = np.median(pandas_iter_times)
        pandas_times.append(elapsed_pandas)
        print(f"  Pandas: median={elapsed_pandas:8.3f} s  (runs: {[round(x,2) for x in pandas_iter_times]})")
    except MemoryError:
        print(f"  Pandas: OOM на {n_rows:,} строк")
        pandas_times.append(None)

    del df, data_spark, data_pandas


# ============================================================
# 3. График
# ============================================================
fig, ax = plt.subplots(figsize=(11, 6))

valid_idx = [i for i, t in enumerate(pandas_times) if t is not None]
valid_n   = [num_users_list[i] for i in valid_idx]
valid_p   = [pandas_times[i]   for i in valid_idx]
valid_s   = [spark_times[i]    for i in valid_idx]

ax.plot(valid_n, valid_s, marker='o', linewidth=2.2, label='Spark',  color='#E25822')
ax.plot(valid_n, valid_p, marker='s', linewidth=2.2, label='Pandas', color='#150458')

ax.set_xscale('log')
ax.set_yscale('log')

ax.set_xlabel('num_users (log scale)', fontsize=13)
ax.set_ylabel('ABTest execution time, s (median, log scale)', fontsize=13)
ax.set_title(
    f'ABTest execution time: Spark vs Pandas\n'
    f'(mega-dataset, target=cltv, treat=[0,1,2], n_iterations={n_iterations})',
    fontsize=14
)
ax.legend(fontsize=12)
ax.grid(True, which='both', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('abtest_spark_vs_pandas_mega.png', dpi=150)
plt.show()


# ============================================================
# 4. CSV
# ============================================================
results_df = pd.DataFrame({
    'num_users': num_users_list,
    'spark_time_s': spark_times,
    'pandas_time_s': [t if t is not None else 'OOM' for t in pandas_times],
    'speedup_pandas_vs_spark': [
        round(tp / ts, 2) if (tp is not None and ts > 0) else None
        for ts, tp in zip(spark_times, pandas_times)
    ],
    'winner': [
        'Pandas' if (tp is not None and tp < ts) else 'Spark'
        for ts, tp in zip(spark_times, pandas_times)
    ]
})

results_df.to_csv('abtest_benchmark_results.csv', index=False)
print("\n=== RESULTS ===")
print(results_df.to_string(index=False))
print("\n✅ Сохранено в abtest_benchmark_results.csv")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ============================================================
# 1. Загрузка данных из CSV
# ============================================================
df = pd.read_csv('abtest_benchmark_results.csv')
print(df)

# Обработка возможных OOM значений в pandas_time_s
df['pandas_numeric'] = pd.to_numeric(df['pandas_time_s'], errors='coerce')

# Убираем строки, где Pandas упал с OOM
df_valid = df.dropna(subset=['pandas_numeric']).reset_index(drop=True)

# ============================================================
# 2. Настройка стиля — делаем красиво
# ============================================================
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': '#333333',
    'axes.linewidth': 1.2,
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
    'figure.dpi': 150,
})

# Красивая палитра
SPARK_COLOR  = '#E8630A'   # тёплый оранжевый (Apache Spark brand)
PANDAS_COLOR = '#130754'   # глубокий синий (Pandas brand)
BG_COLOR     = '#FAFAF7'   # лёгкий кремовый фон

# ============================================================
# 3. Построение графика (ЛИНЕЙНАЯ шкала)
# ============================================================
fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

# --- Основная область под кривыми (shaded) ---
ax.fill_between(
    df_valid['num_users'],
    df_valid['spark_time_s'],
    df_valid['pandas_numeric'],
    where=df_valid['spark_time_s'] < df_valid['pandas_numeric'],
    color=SPARK_COLOR, alpha=0.12,
    label='Spark быстрее',
)
ax.fill_between(
    df_valid['num_users'],
    df_valid['spark_time_s'],
    df_valid['pandas_numeric'],
    where=df_valid['spark_time_s'] >= df_valid['pandas_numeric'],
    color=PANDAS_COLOR, alpha=0.12,
    label='Pandas быстрее',
)

# --- Линии ---
ax.plot(
    df_valid['num_users'], df_valid['spark_time_s'],
    marker='o', markersize=10, markerfacecolor='white',
    linewidth=3, label='Spark', color=SPARK_COLOR,
    markeredgewidth=2.5, markeredgecolor=SPARK_COLOR,
    zorder=5,
)
ax.plot(
    df_valid['num_users'], df_valid['pandas_numeric'],
    marker='s', markersize=10, markerfacecolor='white',
    linewidth=3, label='Pandas', color=PANDAS_COLOR,
    markeredgewidth=2.5, markeredgecolor=PANDAS_COLOR,
    zorder=5,
)

# --- Подписи значений прямо на точках ---
for x, y in zip(df_valid['num_users'], df_valid['spark_time_s']):
    ax.annotate(
        f'{y:.2f}s',
        xy=(x, y), xytext=(0, 16),
        textcoords='offset points',
        ha='center', va='bottom',
        fontsize=10, fontweight='bold', color=SPARK_COLOR,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  edgecolor=SPARK_COLOR, alpha=0.8, linewidth=1),
    )

for x, y in zip(df_valid['num_users'], df_valid['pandas_numeric']):
    ax.annotate(
        f'{y:.2f}s',
        xy=(x, y), xytext=(0, -22),
        textcoords='offset points',
        ha='center', va='top',
        fontsize=10, fontweight='bold', color=PANDAS_COLOR,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  edgecolor=PANDAS_COLOR, alpha=0.8, linewidth=1),
    )

# ============================================================
# 4. Оси и оформление
# ============================================================
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x):,}'.replace(',', ' ')))
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.1f}'))

ax.set_xlabel('Количество пользователей (num_users)', fontsize=14, fontweight='bold', labelpad=12)
ax.set_ylabel('Время выполнения ABTest, секунды', fontsize=14, fontweight='bold', labelpad=12)

# Title с контекстом ABTest
ax.set_title(
    'Производительность ABTest: Spark vs Pandas\n',
    fontsize=18, fontweight='bold', color='#222222', pad=20,
)
ax.text(
    0.5, 1.02,
    r'51 признак  •  treat $\in$ {0, 1, 2}  •  5 итераций  •  Apple M4 Pro',
    transform=ax.transAxes, ha='center', va='bottom',
    fontsize=12, color='#666666', style='italic',
)

# Grid — тонкие пунктирные линии
ax.grid(True, linestyle='--', alpha=0.35, color='#888888', linewidth=0.8)
ax.set_axisbelow(True)

# Y начинается с 0
y_max = max(df_valid['spark_time_s'].max(), df_valid['pandas_numeric'].max())
ax.set_ylim(0, y_max * 1.25)

# Красивый legend
legend = ax.legend(
    loc='upper left',
    frameon=True,
    fancybox=True,
    shadow=False,
    framealpha=0.95,
    edgecolor='#CCCCCC',
    fontsize=12,
    title='Backend / Зона доминирования',
    title_fontsize=11,
)
legend.get_title().set_fontweight('bold')

# ============================================================
# 5. Добавляем инсайт-блок (crossover point)
# ============================================================
crossover_idx = None
for i in range(len(df_valid) - 1):
    s1, s2 = df_valid.iloc[i]['spark_time_s'],     df_valid.iloc[i+1]['spark_time_s']
    p1, p2 = df_valid.iloc[i]['pandas_numeric'],   df_valid.iloc[i+1]['pandas_numeric']
    if p1 < s1 and p2 > s2:
        crossover_idx = i
        break

# if crossover_idx is not None:
#     cx = df_valid.iloc[crossover_idx + 1]['num_users']
#     ax.axvline(
#         x=cx, color='#555555', linestyle=':', linewidth=1.5, alpha=0.7,
#     )
#     ax.annotate(
#         f'← Точка кроссовера\n     ~{int(cx):,} пользователей',
#         xy=(cx, y_max * 0.85), xytext=(cx * 0.5, y_max * 1.05),
#         arrowprops=dict(arrowstyle='->', color='#555555', lw=1.5),
#         ha='center', fontsize=11, color='#444444',
#         bbox=dict(boxstyle='round,pad=0.5', facecolor='#FFF8DC',
#                   edgecolor='#AAAAAA', linewidth=1.2),
#     )
# else:
#     # Если кроссовера нет (один всегда быстрее), добавим комментарий
#     ax.annotate(
#         'Spark быстрее\nна всех объёмах' if df_valid.iloc[-1]['spark_time_s'] < df_valid.iloc[-1]['pandas_numeric']
#         else 'Pandas быстрее\nна всех объёмах',
#         xy=(df_valid.iloc[-1]['num_users'], df_valid.iloc[-1]['pandas_numeric']),
#         xytext=(df_valid.iloc[-1]['num_users'] * 0.6, y_max * 1.10),
#         arrowprops=dict(arrowstyle='->', color='#555555', lw=1.5),
#         ha='center', fontsize=11, color='#444444',
#         bbox=dict(boxstyle='round,pad=0.5', facecolor='#FFF8DC',
#                   edgecolor='#AAAAAA', linewidth=1.2),
#     )

plt.tight_layout()
plt.savefig('abtest_benchmark_linear_beautiful.png', dpi=200, bbox_inches='tight',
            facecolor=BG_COLOR)
plt.show()

print("✅ Сохранено: abtest_benchmark_linear_beautiful.png")

In [ ]:
   num_users  spark_time_s  pandas_time_s  speedup_pandas_vs_spark  winner
0      10000      4.582543       1.544527                     0.34  Pandas
1      50000      4.366532       4.404127                     1.01   Spark
2     100000      4.383894       7.671143                     1.75   Spark
3     500000      5.736172      33.714123                     5.88   Spark
4    1000000      7.592676      67.481100                     8.89   Spark

## AA test
Then we run the experiment on our prepared dataset, wrapped into ExperimentData. In this case we select one of the pre-assembled pipeline, AA_TEST.
We can set the number of iterations for simple execution. In this case the random states are the numbers of each iteration.

In [ ]:
test = AATest(n_iterations=10)
result = test.execute(data)

In [ ]:
result.resume

**Interpretation of AA test results**

Each row in the table corresponds to a target feature being tested for equality between the control and test groups. Two statistical tests are used:

- **TTest**: tests if means are statistically different.
- **KSTest**: tests if distributions differ.

The `OK` / `NOT OK` labels show whether the difference is statistically significant. A `NOT OK` result indicates a possible imbalance.

Typical threshold:
- If p-value < 0.05 → `NOT OK` (statistically significant difference)
- If p-value ≥ 0.05 → `OK` (no significant difference)

If any metric has a `NOT OK` status in the `AA test` column, it means at least one iteration showed significant difference.


In [ ]:
result.aa_score

**Interpreting `aa_score`**

This output shows p-values and the overall pass/fail status for each test type and feature. A high p-value (close to 1.0) means the test passed — the groups are similar.

- `score`: p-value of the statistical test.
- `pass`: True if no iterations showed significant differences.

Note: Even if the average p-value is high, the `pass` might still be False if at least one of the iterations had a p-value < 0.05.


In [ ]:
result.best_split

**About `best_split`**

This shows the best found split of the dataset, where control and test groups are as similar as possible in terms of target metrics.

You can use this split for future modeling or as a validation check before proceeding to actual experiments.


In [ ]:
result.best_split_statistic

**Understanding `best_split_statistic`**

This table contains detailed statistics for the best (most balanced) split found across all iterations. You can compare:

- Mean values in control vs test group.
- Absolute and relative differences.
- p-values for both tests.

Ideally, all rows should have `OK` in both TTest and KSTest columns, and small difference values (<1%).

In [ ]:
result.experiments

# AA Test with random states

We can also adjust some of the preset parameters of the experiment by assigning them to the respective params of the experiment. I.e. here we set the range of the random states we want to run our AA test for. 

In [ ]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    }, 
    data=create_test_data(),
    session=sp_s,
    backend=BackendsEnum.spark
)
data

In [ ]:
test = AATest(random_states=[56, 72, 2, 43])
result = test.execute(data)

In [ ]:
result.resume

In [ ]:
result.aa_score

In [ ]:
result.best_split

In [ ]:
result.best_split_statistic

In [ ]:
result.experiments

# AA Test with stratification

Depending on your requirements it is possible to stratify the data. You can set `stratification=True` and `StratificationRole` in `Dataset` to run it with stratification.

Stratified AA tests ensure that both groups (control/test) have the same proportions of categories (e.g. same % of genders or regions). This prevents imbalances in categorical features that can distort results.

Make sure to assign `StratificationRole` to relevant columns in your dataset before enabling stratification.

In [ ]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    }, 
    data=create_test_data(),
    session=sp_s,
    backend=BackendsEnum.spark
)
data

In [ ]:
test = AATest(random_states=[56, 72, 2, 43], stratification=True)
result = test.execute(data)

In [ ]:
result.resume

In [ ]:
result.aa_score

In [ ]:
result.best_split

In [ ]:
result.best_split_statistic

In [ ]:
result.experiments

# AA Test by samples 

Depending on your requirements and size of data it is possible to estimate AA test on samples the data. You can set `sample_size=size` to run it. 

In [ ]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    },
    data=create_test_data(),
    session=sp_s,
    backend=BackendsEnum.spark
)
data

In [ ]:
test = AATest(n_iterations=10, sample_size=0.3)
result = test.execute(data)

In [ ]:
result.resume

In [ ]:
result.aa_score

In [ ]:
result.best_split

In [ ]:
result.best_split_statistic

In [ ]:
result.experiments

# AATest with Target Role for a categorical feature

It is possible to assign Target Role to categorical features. A categorical feature can also be the target or outcome variable. In this case, the Chi-square test is added to the pipeline of AATest.

In [ ]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "treat": TreatmentRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": TargetRole(str)
    }, data=create_test_data(),
)
data

In [ ]:
test = AATest(n_iterations=10)
result = test.execute(data)

In [ ]:
result.resume

In [ ]:
result.aa_score

In [ ]:
result.best_split

In [ ]:
result.best_split_statistic

In [ ]:
result.experiments

# AATest with unequal group sizes

AATest can be performed to get a split with unequal the groups of different sizes by using `unequal_size` argument. Also Whelch correction can be applied by adding `t_test_equal_vat=False` argument while initiating AATest instance.

In [ ]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    },
    data=create_test_data(),
    session=sp_s,
    backend=BackendsEnum.spark
)
data

In [ ]:
test = AATest(n_iterations=10, control_size=0.3, t_test_equal_var=False)
result = test.execute(data)

In [ ]:
result.best_split.data.groupby("split").agg("count")

In [ ]:
result.best_split_statistic

# AAnTest

AAnTest is an extension of AATest that allows to split the dataset into several test groups, additionally to the control group.

In [ ]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    },
    data=create_test_data(),
    session=sp_s,
    backend=BackendsEnum.spark
)
data

In [ ]:
test = AATest(groups_sizes=[0.3, 0.2, 0.2, 0.3])
result = test.execute(data)

In [ ]:
result.best_split.data.groupby("split").agg("count")

In [ ]:
result.best_split_statistic

# AATest with partially pre-defined groups

Certain users can be pre-assigned to either the test or the control group, so that they are not randomly assigned. This can be done using the `ConstGroupRole` role. In order to pre-assign users to the control group they should have a value of `control`, and in the test group they should have a value of `test` in the column with the role `ConstGroupRole`. Users that are not pre-assigned to either the control or the test group should have `None`, so that they will be assigned randomly.

In [ ]:
pd_data= create_test_data()
pd_data.loc[pd_data["treat"]==0, "const_grp"] = "control"
pd_data.loc[pd_data["treat"]==1, "const_grp"] = "test"
pd_data.loc[2000:, "const_grp"] = None

data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "const_grp": ConstGroupRole(str),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
        "industry": TargetRole(str),
    }, data=pd_data,
    session=sp_s,
    backend=BackendsEnum.spark
)
data

In [ ]:
test = AATest(n_iterations=1)
result = test.execute(data)

In [ ]:
result.resume

In [ ]:
result.best_split

## Common issues and tips

- **Missing roles**: Make sure all target variables are assigned `TargetRole`. Columns without roles may cause silent failure.
- **Stratification**: If your dataset contains categorical features (e.g. `gender`, `region`) that may affect the outcome, use `StratificationRole` and enable `stratification=True` in `AATest(...)`.
- **Imbalanced categories**: If some categories have too few samples, stratified splits may become unstable. Consider filtering or merging rare categories.
- **Random fluctuations**: On small datasets, it's normal to see occasional `NOT OK` results. Use more iterations (e.g. `n_iterations=50`) for stability.
- **Missing values**: NaNs in stratification columns may be treated as separate categories. Clean or fill missing values before stratified AA tests.